In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
Dir_github = '/work/desai-lab/xuanyang/Project/ABC/github'
Dir_dataset = '/work/desai-lab/ABC_dataset'
Dir_preprocess = os.path.join(Dir_dataset,'derivatives','fmriprep')


# Make master sheet

In [2]:
df1 = pd.read_csv(os.path.join(Dir_dataset,'codes','Eprime_merged1_1197.csv'))
df2 = pd.read_csv(os.path.join(Dir_dataset,'codes','Eprime_merged2_1198_1361.csv'))
dfs_eprime = pd.concat([df1,df2],ignore_index=True)
dfs_eprime.loc[dfs_eprime['Subject']==137,'Subject'] = 1037

# eprime
df_master_eprime = dfs_eprime.groupby(['Subject','Session']).count().reset_index()
df_master_eprime = df_master_eprime[['Subject','Session']]
df_master_eprime['Fam_eprime'] = 1
df_master_eprime.rename({'Subject':'subid','Session':'visit'},axis=1,inplace=True)

# fmri
List_folders = [x for x in os.listdir(Dir_preprocess) if '.html' not in x and 'sub' in x]
df_master_fmri = []
for subID in List_folders:
    List_visits = [x for x in os.listdir(os.path.join(Dir_preprocess,subID)) if 'visit' in x]
    if len(List_visits)>0:
        for visit in List_visits:
            List_files = os.listdir(os.path.join(Dir_preprocess,subID,visit,'func'))
            
            for task in ['rest','Passage','Fam']:
                List_imgs = [x for x in List_files if task in x and 'preproc_bold.nii.gz' in x]
                if len(List_imgs)>0:
                    df_master_fmri.append(pd.DataFrame({'subid':subID,'visit':visit,'task':task+'_fmri','fmri':1},index=[0]))
df_master_fmri = pd.concat(df_master_fmri,ignore_index=True)
df_master_fmri = df_master_fmri.pivot(index=['subid','visit'],columns='task',values='fmri').reset_index()

df_master_fmri['subid'] = df_master_fmri['subid'].str.replace('sub-','').astype(int)
df_master_fmri['visit'] = df_master_fmri['visit'].str.replace('ses-visit0','').astype(int)

# behavioral questionnaire data
df_behav = pd.read_csv(os.path.join(Dir_dataset,'codes','ABCStudyBehavioral_DATA_2024-08-13_0912.csv'))
df_behav = df_behav.loc[~df_behav['confirm_abcid'].isna()].reset_index(drop=True)
df_behav['subid'] = df_behav['confirm_abcid'].str.split('ABC',expand=True)[1]
df_behav = df_behav[['subid','age_dayofday1testing','sex','first_language','other_language']]
df_behav.rename({'age_dayofday1testing':'age'},axis=1,inplace=True)
df_behav['subid'] = df_behav['subid'].astype(int)

# combine
df_master = df_master_fmri.merge(df_master_eprime,on=['subid','visit'],how='outer')

df_master = df_master.merge(df_behav,on=['subid'],how='outer')

df_master = df_master[['subid','visit','Fam_fmri','Fam_eprime','age','sex']]
df_master = df_master.dropna()
df_master = df_master.loc[df_master['visit']==1]

# export age range
for age_min in [20,40,60]:
    df = df_master.loc[df_master['age']>=age_min].loc[df_master['age']<age_min+20].loc[df_master['visit']==1]
    print(f"{age_min}-{age_min+20} yrs: {df.shape[0]} subjects")
# df_master['visit'] = df_master['visit'].astype(int)
df_master

20-40 yrs: 82 subjects
40-60 yrs: 27 subjects
60-80 yrs: 77 subjects


,subid,visit,Fam_fmri,Fam_eprime,age,sex
7,1005,1.0,1.0,1.0,65.0,2.0
9,1006,1.0,1.0,1.0,61.0,2.0
10,1007,1.0,1.0,1.0,69.0,2.0
11,1008,1.0,1.0,1.0,77.0,2.0
12,1009,1.0,1.0,1.0,75.0,2.0
...,...,...,...,...,...,...
235,1311,1.0,1.0,1.0,20.0,2.0
239,1318,1.0,1.0,1.0,71.0,2.0
240,1320,1.0,1.0,1.0,52.0,2.0
241,1321,1.0,1.0,1.0,34.0,2.0


# Check Motion

In [3]:
List_motions = ['trans_x','trans_y','trans_z','rot_x','rot_y','rot_z']
for i,row in df_master.iterrows():
    subid = int(row['subid'])
    visit = int(row['visit'])
    path = os.path.join(Dir_preprocess,f"sub-{subid}",f"ses-visit0{visit}",'func')
    file = os.path.join(path,f"sub-{subid}_ses-visit0{visit}_task-Fam_desc-confounds_timeseries.tsv")
    df = pd.read_csv(file,sep='\t')
    df_master.loc[i,List_motions] = df[List_motions].max()
    df_master.loc[i,'motion_max'] = df[List_motions].max().max()

for cutoff in [1,1.5,3]:
    df = df_master.loc[df_master['motion_max']<=cutoff]
    print(f"<{cutoff}: {df.shape[0]}")

<1: 120
<1.5: 155
<3: 183


In [249]:
df_output = df_master.loc[df_master['motion_max']<=1]
path_output = os.path.join(Dir_github,'derivatives','test_Fam')
if not os.path.exists(path_output):
    os.makedirs(path_output)
df_output.to_csv(os.path.join(path_output,'df_master_Fam120.csv'),index=False)
df_output

,subid,visit,Fam_fmri,Fam_eprime,age,sex,trans_x,trans_y,trans_z,rot_x,rot_y,rot_z,motion_max
7,1005,1.0,1.0,1.0,65.0,2.0,0.041617,0.267869,0.463099,0.007059,0.003916,0.003123,0.463099
10,1007,1.0,1.0,1.0,69.0,2.0,0.383059,0.671302,0.386057,0.009304,0.001248,0.003724,0.671302
11,1008,1.0,1.0,1.0,77.0,2.0,0.075486,0.203145,0.118612,0.005849,0.000214,0.001656,0.203145
12,1009,1.0,1.0,1.0,75.0,2.0,0.000065,0.349682,0.215196,0.003958,0.000177,0.000196,0.349682
20,1018,1.0,1.0,1.0,60.0,2.0,0.026452,0.412780,0.433625,0.003764,0.005943,0.000393,0.433625
...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,1302,1.0,1.0,1.0,23.0,1.0,0.565818,0.318678,0.295166,0.004322,0.001432,0.012690,0.565818
231,1305,1.0,1.0,1.0,61.0,2.0,0.061026,0.375817,0.324899,0.005508,0.000556,0.000923,0.375817
232,1306,1.0,1.0,1.0,28.0,2.0,0.485715,0.411004,0.707648,0.018751,0.001389,0.016267,0.707648
235,1311,1.0,1.0,1.0,20.0,2.0,0.174499,0.285192,0.111143,0.003649,0.008804,0.005942,0.285192


In [252]:
dfs_eprime.to_csv(os.path.join(path_output,'dfs_eprime_Fam_by1361.csv'),index=False)